# ST-EEGFormer Pretrained + LoRA

This notebook uses the official ST-EEGFormer repo plus the Hugging Face pretrained encoder weights, then adds LoRA adapters to attention blocks and a binary classifier head for the Congruent vs Incongruent Pure task.


In [8]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt

# %pip install -q mne
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Prefer either the Colab-mounted path or the local repo path
try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive")
    COLAB_SRC = Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if COLAB_SRC.exists():
        SRC_DIR = COLAB_SRC
    else:
        IN_COLAB = False
        NOTEBOOK_DIR = Path.cwd()
        SRC_DIR = NOTEBOOK_DIR.parent.parent / "src"
except Exception:
    IN_COLAB = False
    NOTEBOOK_DIR = Path.cwd()
    SRC_DIR = NOTEBOOK_DIR.parent.parent / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import BinaryTaskConfig, RawEpochSpec, detect_project_paths, load_raw_epoch_dataset
from br_analysis.pretrained import BackboneClassifier, prepare_lora_backbone, load_backbone_weights
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
)
from br_analysis.torch_models import count_trainable_parameters

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")

from br_analysis import electrode_distance_adjacency, temporal_band_features
from br_analysis.torch_models import STGNNChebConv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ModuleNotFoundError: No module named 'br_analysis.pretrained'

In [10]:
import sys
import subprocess
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
%pip install -q mne
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (
    Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if "google.colab" in sys.modules
    else NOTEBOOK_DIR.parent.parent / "src"
)

try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=True)
except ImportError:
    IN_COLAB = False

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import BinaryTaskConfig, RawEpochSpec, detect_project_paths, load_raw_epoch_dataset
from br_analysis.pretrained import BackboneClassifier, prepare_lora_backbone, load_backbone_weights
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
)
from br_analysis.torch_models import count_trainable_parameters

# try:
#     from google.colab import drive

#     IN_COLAB = True
#     drive.mount("/content/drive")
# except ImportError:
#     IN_COLAB = False

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")

Mounted at /content/drive


ModuleNotFoundError: No module named 'br_analysis.pretrained'

## Task And Environment


In [ ]:
# ST-EEGFormer needs the Hugging Face downloader and the official repo code.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "safetensors", "timm"], check=True)

TASK_TAG = "c_vs_ip_steegformer_pretrained_lora"
paths = detect_project_paths("c_vs_i_steegformer_pretrained_lora", in_colab=IN_COLAB)
SELECTED_SUBJECTS_CSV = paths.analysis_root / "01_exploracio" / "tables" / "subjectes_seleccionats.csv"
MAX_SUBJECTS = 22

REPO_DIR = paths.analysis_dir / "external" / "STEEGFormer"
WEIGHTS_DIR = paths.cache_dir / "steegformer_weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/LiuyinYang1101/STEEGFormer", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    from models_mae_eeg import mae_vit_base_patch16
except ImportError as exc:
    raise ImportError("Could not import models_mae_eeg from the official ST-EEGFormer repo.") from exc

from huggingface_hub import hf_hub_download

TASK = BinaryTaskConfig(
    name=TASK_TAG,
    class_0="Congruent",
    class_1="Incongruent_Pur",
    event_codes={"Congruent": [1, 2], "Incongruent_Pur": [3, 4, 5, 6]},
    selected_subjects_csv=SELECTED_SUBJECTS_CSV,
    max_subjects=MAX_SUBJECTS,
    undersample_classes=True,
    analysis_window=(0.0, 1.5),
    random_seed=RANDOM_SEED,
)

print(f"Projecte: {paths.project_dir}")
print(f"Resultats: {paths.analysis_dir}")
print(f"Repo ST-EEGFormer: {REPO_DIR}")
print(f"Weights dir: {WEIGHTS_DIR}")

## Load Raw Epochs


In [ ]:
spec = RawEpochSpec(picks=None, bandpass=(1.0, 40.0), resample_sfreq=128.0)
dataset = load_raw_epoch_dataset(paths, TASK, spec)
X = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y, dtype=np.int64)
ch_names = list(dataset.channel_names)
times = dataset.times

# Pad/truncate to the ST-EEGFormer reference length (6s @ 128Hz = 768 samples).
TARGET_LEN = 768
if X.shape[-1] < TARGET_LEN:
    X = np.pad(X, ((0, 0), (0, 0), (0, TARGET_LEN - X.shape[-1])), mode="constant")
else:
    X = X[:, :, :TARGET_LEN]

print(f"X shape: {X.shape}")
print(f"Classes: {np.bincount(y)}")

plt.figure(figsize=(5, 4))
sns.countplot(x=np.where(y == 0, TASK.class_0, TASK.class_1))
plt.title("Class distribution")
plt.xlabel("Class")
plt.tight_layout()
plt.savefig(paths.figures_dir / f"class_distribution_{TASK_TAG}.png", dpi=150, bbox_inches="tight")
plt.show()

## Build Pretrained Model


In [ ]:
# Download the official safetensors checkpoint and build the base encoder backbone.
checkpoint_path = Path(
    hf_hub_download(
        repo_id="eugenehp/ST-EEGFormer",
        filename="ST-EEGFormer_base_encoder.safetensors",
        local_dir=str(WEIGHTS_DIR),
        local_dir_use_symlinks=False,
    )
)
print(f"Checkpoint: {checkpoint_path}")

backbone = mae_vit_base_patch16()
load_backbone_weights(backbone, checkpoint_path, strict=False)
prepare_lora_backbone(
    backbone,
    include_substrings=("attn",),
    exclude_substrings=("head", "classifier", "decoder", "patch_embed", "mask_token"),
    r=8,
    alpha=16.0,
    dropout=0.05,
    train_classifier=False,
)
model = BackboneClassifier(backbone, n_classes=2, example_input=torch.from_numpy(X[:1]))
trainable, total = count_trainable_parameters(model)
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## Train


In [ ]:
def make_model():
    backbone = mae_vit_base_patch16()
    load_backbone_weights(backbone, checkpoint_path, strict=False)
    prepare_lora_backbone(
        backbone,
        include_substrings=("attn",),
        exclude_substrings=("head", "classifier", "decoder", "patch_embed", "mask_token"),
        r=8,
        alpha=16.0,
        dropout=0.05,
        train_classifier=False,
    )
    model = BackboneClassifier(backbone, n_classes=2, example_input=torch.from_numpy(X[:1]))
    trainable, total = count_trainable_parameters(model)
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


train_cfg = TorchTrainConfig(
    epochs=30,
    batch_size=12,
    lr=2e-4,
    weight_decay=1e-4,
    patience=6,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
)

metrics, history = cross_validate_torch_model(X, y, make_model, train_cfg)
metrics_path = paths.tables_dir / f"cv_metrics_{TASK_TAG}.csv"
history_path = paths.tables_dir / f"training_history_{TASK_TAG}.csv"
metrics.to_csv(metrics_path, index=False)
history.to_csv(history_path, index=False)
print(f"Saved metrics: {metrics_path}")
print(f"Saved history: {history_path}")
metrics

## Visualize Training


In [ ]:
plot_fold_metrics(metrics, output_file=paths.figures_dir / f"fold_metrics_{TASK_TAG}.png")
plot_torch_history(history, output_file=paths.figures_dir / f"training_history_{TASK_TAG}.png")

summary = metrics.drop(columns=["fold"]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
summary_path = paths.tables_dir / f"summary_{TASK_TAG}.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary